# Advanced CLIP Experiment — Heatmap on Search Terms



### Copyright notice
HSLU DFK | Colabor: Zoom In Zoom Out | 2026 Thomas Knüsel, [MIT License](LICENSE). 
Feel free to modify and reuse the content. 

---

This notebook uses OpenAI's **CLIP** model to localize where a free-text concept
appears inside an image. It slides patches across the image, embeds each patch
together with your text prompt, and renders the cosine similarity as a heatmap.


## 1. Install dependencies

Installs PyTorch, OpenCV, CLIP, ipywidgets and the few helpers we need. Run once
per environment — it's a no-op if everything is already present.


In [ ]:
import sys
!{sys.executable} -m pip install -q torch torchvision opencv-python ftfy regex tqdm matplotlib pillow ipywidgets
!{sys.executable} -m pip install -q git+https://github.com/openai/CLIP.git

## 2. Imports and model loading

Imports the libraries, picks the best available device (CUDA → Apple MPS → CPU),
and loads the **ViT-B/32** CLIP weights. The model stays in memory for the rest
of the session, so you only pay this cost once.


In [ ]:
import io
import time
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import clip
import ipywidgets as widgets
from IPython.display import display

# Device selection (CUDA > Apple MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

print("Loading CLIP model...")
model, preprocess = clip.load("ViT-B/32", device=device)
print("Model loaded successfully!")



## 3. Upload your image

Click the button below and choose any JPEG or PNG. The file is held in memory
inside the widget — nothing is written to disk yet.


In [ ]:
uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(uploader)
print("Click the button above and choose an image, then run the next cell.")

## 4. Load and preview the image

Reads the bytes from the upload widget into a PIL `Image`, converts to RGB and
shows a preview so you can confirm the right file was picked.


In [ ]:
if not uploader.value:
    raise RuntimeError("No file uploaded yet — use the button in the previous cell first.")

# ipywidgets 8+ returns a tuple of dicts; older versions return a dict keyed by filename
if isinstance(uploader.value, tuple):
    entry = uploader.value[0]
    image_path = entry['name']
    content = entry['content']
else:
    image_path = next(iter(uploader.value))
    content = uploader.value[image_path]['content']

image = Image.open(io.BytesIO(content)).convert("RGB")

plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.title("Input Image")
plt.axis('off')
plt.show()
print(f"Analyzing image: {image_path}, size: {image.width}x{image.height}")

## 5. Define the heatmap function

`enhanced_concept_heatmap(image, concept, ...)` is the core routine:

- Resizes large images to a max dimension of 512 px for speed.
- Slides square patches across the image at the given `patch_sizes` and `strides`.
- For each patch, embeds it with CLIP and compares against three prompt
  variations of your concept (`"X"`, `"a X"`, `"the X"`) — taking the max
  similarity makes the result less sensitive to prompt phrasing.
- Accumulates similarities into a 2-D map, normalizes by the 5th/95th
  percentiles, applies a non-linear contrast curve and a Gaussian blur, then
  resizes back to the original image dimensions.

Just defining the function here — nothing runs yet.


In [ ]:
# ============================================================
# CELL 5 — Heatmap function
# ============================================================
def enhanced_concept_heatmap(image, concept, patch_sizes=[64, 96, 128], strides=[16, 32, 48]):
    # Resize large images for performance
    width, height = image.size
    max_dim = 512
    if max(width, height) > max_dim:
        scale = max_dim / max(width, height)
        new_size = (int(width * scale), int(height * scale))
        img = image.resize(new_size, Image.LANCZOS)
        print(f"Resized to {new_size} for processing")
    else:
        img = image

    heatmap = np.zeros((img.height, img.width))
    counts = np.zeros((img.height, img.width))

    prompt_variations = [concept, f"a {concept}", f"the {concept}"]
    text_tokens = clip.tokenize(prompt_variations).to(device)

    for patch_size, stride in zip(patch_sizes, strides):
        print(f"  Processing with patch size {patch_size}px, stride {stride}px")

        for y in range(0, img.height - patch_size + 1, stride):
            for x in range(0, img.width - patch_size + 1, stride):
                patch = img.crop((x, y, x + patch_size, y + patch_size))
                patch_tensor = preprocess(patch).unsqueeze(0).to(device)

                with torch.no_grad():
                    image_features = model.encode_image(patch_tensor)
                    image_features /= image_features.norm(dim=-1, keepdim=True)

                    text_features = model.encode_text(text_tokens)
                    text_features /= text_features.norm(dim=-1, keepdim=True)

                    similarities = (100.0 * image_features @ text_features.T)
                    max_similarity = float(similarities.max().item())

                weight = 1.0 / patch_size
                heatmap[y:y+patch_size, x:x+patch_size] += max_similarity * weight
                counts[y:y+patch_size, x:x+patch_size] += weight

    heatmap = np.divide(heatmap, counts, out=np.zeros_like(heatmap), where=counts != 0)

    # Percentile-based normalization
    p_low, p_high = np.percentile(heatmap, [5, 95])
    heatmap = np.clip((heatmap - p_low) / (p_high - p_low + 1e-8), 0, 1)

    # Non-linear contrast
    heatmap = np.power(heatmap, 1.5)

    # Adaptive smoothing
    kernel_size = max(3, min(31, int(min(img.width, img.height) / 20)))
    if kernel_size % 2 == 0:
        kernel_size += 1
    heatmap = cv2.GaussianBlur(heatmap, (kernel_size, kernel_size), 0)

    if img.size != image.size:
        heatmap = cv2.resize(heatmap, (image.width, image.height))

    return heatmap

## 6. Enter your search terms

Type one concept per line in the box below, then click **Run analysis**. You can
edit the list and click the button again as often as you like without
re-running the earlier cells.

**Tips for writing good CLIP prompts**

- **Use natural noun phrases**, not single keywords: `"a person wearing a red
  hat"` works better than `"hat"`.
- **Be visually concrete**. CLIP responds to what things *look* like, not to
  abstract attributes. `"rusty metal surface"` beats `"old"`.
- **Compose with relations**: `"man riding a dinosaur"`, `"dog next to a
  bicycle"`, `"hand holding a cup"` — CLIP handles simple spatial/action
  relations surprisingly well.
- **Try contrasting prompts** for the same image (e.g. `"face"` vs
  `"background"`, or `"foreground object"` vs `"sky"`) — the differences are
  often more informative than any single heatmap.
- **Avoid negations** like `"not a cat"`. CLIP largely ignores them.
- **Iterate.** If a concept lights up the wrong region, rephrase: `"green
  dinosaur skin"` instead of `"green dinosaur"`, or add context: `"the head of
  a dinosaur"`.

The two sliders let you trade speed against spatial detail: smaller patches and
strides give a finer heatmap but take much longer.


In [ ]:
# Interactive concept input — type your search terms below, one per line,
# then click "Run analysis". You can change the terms and click again at any time
# without re-executing earlier cells.

concepts_input = widgets.Textarea(
    value="man riding dinosaur\nman's face\ngreen dinosaur\ndinosaur head\nhat",
    placeholder="Type one concept per line...",
    description="Concepts:",
    layout=widgets.Layout(width="100%", height="160px"),
    style={"description_width": "initial"},
)

patch_size_slider = widgets.IntSlider(
    value=96, min=48, max=160, step=16,
    description="Patch size (px):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="60%"),
)

stride_slider = widgets.IntSlider(
    value=32, min=8, max=64, step=8,
    description="Stride (px):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="60%"),
)

run_button = widgets.Button(
    description="Run analysis",
    button_style="primary",
    icon="play",
)

output_area = widgets.Output()

# Stores results so later cells / further inspection can use them.
results = {}

def _on_run_clicked(_btn):
    output_area.clear_output(wait=True)
    raw = concepts_input.value.strip()
    if not raw:
        with output_area:
            print("Please enter at least one concept.")
        return

    concepts = [line.strip() for line in raw.splitlines() if line.strip()]
    psize = patch_size_slider.value
    stride = stride_slider.value

    with output_area:
        for concept in concepts:
            print(f"\nAnalyzing concept: '{concept}'")
            start = time.time()

            heatmap = enhanced_concept_heatmap(
                image, concept,
                patch_sizes=[psize], strides=[stride],
            )

            heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
            heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

            results[concept] = {"heatmap": heatmap, "colored": heatmap_colored}

            plt.figure(figsize=(15, 5))
            plt.subplot(1, 3, 1)
            plt.imshow(image); plt.title("Original Image"); plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(heatmap_colored); plt.title(f"Heatmap: '{concept}'"); plt.axis("off")

            plt.subplot(1, 3, 3)
            overlay = cv2.addWeighted(np.array(image), 0.7, heatmap_colored, 0.3, 0)
            plt.imshow(overlay); plt.title("Overlay"); plt.axis("off")

            plt.tight_layout()
            plt.show()

            print(f"Processed in {time.time() - start:.1f} seconds")

        print("\nAnalysis complete!")

run_button.on_click(_on_run_clicked)

display(widgets.VBox([
    concepts_input,
    patch_size_slider,
    stride_slider,
    run_button,
    output_area,
]))
